In [1]:
#!/usr/bin/env python3
"""
=============================================================================
PROPORTION FIX ONLY — donor_id 추출 + proportion 재계산 + CSV 저장
Colab 단일 셀 실행
=============================================================================
"""
!pip install -q scanpy leidenalg pyscenic gseapy openpyxl

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import scipy.sparse as sp
import os
import warnings
warnings.filterwarnings('ignore')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.3/605.3 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# 1. LOAD + donor_id 추출
# ============================================================
H5AD_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
CSV_DIR = '/content/drive/MyDrive/ITLAS/results/task3_donor_validation/csv'
os.makedirs(CSV_DIR, exist_ok=True)

print("Loading...")
adata = sc.read_h5ad(H5AD_PATH)
print(f"Loaded: {adata.shape[0]:,} cells")

# donor_id: GSM5519467_P190604_Blood_1 → P190604
CSV_DIR = '/content/drive/MyDrive/ITLAS/results/task3_donor_validation/csv'
os.makedirs(CSV_DIR, exist_ok=True)
adata.obs['donor_id'] = adata.obs['sample'].astype(str).apply(
    lambda x: x.split('_')[1] if len(x.split('_')) >= 2 else x)

# sample_id: globally unique
adata.obs['sample_id'] = adata.obs['Stage'].astype(str) + '_' + adata.obs['donor_id'].astype(str)

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']
LINEAGES = ['NK', 'CD8_T', 'CD4_T', 'Myeloid', 'B', 'PlasmaB']

# Quick check
print(f"Donors: {adata.obs['donor_id'].nunique()}")
for s in STAGE_ORDER:
    n = adata.obs[adata.obs['Stage']==s]['sample_id'].nunique()
    print(f"  {s}: n={n} sample_ids")

Loading...
Loaded: 243,000 cells
Donors: 23
  NL: n=6 sample_ids
  IT: n=6 sample_ids
  IA: n=5 sample_ids
  AR: n=3 sample_ids
  CR: n=3 sample_ids


In [4]:
# ============================================================
# 2. HELPER
# ============================================================
def donor_test(vals_a, vals_b):
    """Direct array input — no DataFrame filtering."""
    if len(vals_a) < 2 or len(vals_b) < 2:
        return {'n_a':len(vals_a),'n_b':len(vals_b),'change':'N/A',
                'consist':0,'total':0,'p':1.0}
    _, p_gt = mannwhitneyu(vals_a, vals_b, alternative='greater')
    _, p_lt = mannwhitneyu(vals_a, vals_b, alternative='less')
    if p_gt < p_lt:
        consist = sum(1 for a in vals_a for b in vals_b if a > b)
        p_use = p_gt
    else:
        consist = sum(1 for a in vals_a for b in vals_b if a < b)
        p_use = p_lt
    total = len(vals_a) * len(vals_b)
    ma, mb = np.mean(vals_a), np.mean(vals_b)
    ch = f"{(mb-ma)/ma*100:+.1f}%" if ma > 0 else "N/A"
    return {'n_a':len(vals_a),'n_b':len(vals_b),'change':ch,
            'consist':consist,'total':total,'p':p_use}

In [5]:
# ============================================================
# 3. COMPUTE PROPORTION PER SAMPLE_ID
# ============================================================
print("\n" + "="*60)
print("LINEAGE PROPORTIONS — FIXED")
print("="*60)

# Total cells per sample_id
totals = adata.obs.groupby('sample_id').size().to_dict()
# Stage per sample_id
stage_map = adata.obs.groupby('sample_id')['Stage'].first().to_dict()

all_stats = []

for lineage in LINEAGES:
    # Lineage cells per sample_id
    lin_counts = adata.obs[adata.obs['major_lineage']==lineage].groupby('sample_id').size().to_dict()

    # Compute proportion for every sample_id
    records = []
    for sid in totals:
        lc = lin_counts.get(sid, 0)
        records.append({
            'sample_id': sid,
            'Stage': stage_map[sid],
            'lineage': lineage,
            'lin_cells': lc,
            'total_cells': totals[sid],
            'pct': lc / totals[sid] * 100
        })
    df = pd.DataFrame(records)

    print(f"\n--- {lineage} ---")
    for stage in STAGE_ORDER:
        sd = df[df['Stage']==stage]['pct']
        if len(sd) > 0:
            print(f"  {stage}: {sd.mean():.1f}% ± {sd.std():.1f}% (n={len(sd)})")

    for sa, sb in [('NL','IT'),('NL','CR'),('IT','IA'),('IA','AR')]:
        va = df[df['Stage']==sa]['pct'].values
        vb = df[df['Stage']==sb]['pct'].values
        r = donor_test(va, vb)
        sig = " **" if r['p']<0.05 else " *" if r['p']<0.1 else ""
        print(f"  {sa}→{sb}: {r['change']}, {r['consist']}/{r['total']} "
              f"(n={r['n_a']}v{r['n_b']}), p={r['p']:.3f}{sig}")
        all_stats.append({
            'lineage':lineage,'comparison':f'{sa}_vs_{sb}',
            'n_a':r['n_a'],'n_b':r['n_b'],'change':r['change'],
            'consistency':r['consist'],'total_pairs':r['total'],
            'p_value':r['p'],'sig_05':r['p']<0.05})


LINEAGE PROPORTIONS — FIXED

--- NK ---
  NL: 37.0% ± 8.9% (n=6)
  IT: 27.7% ± 9.7% (n=6)
  IA: 18.6% ± 6.9% (n=5)
  AR: 20.7% ± 3.7% (n=3)
  CR: 12.7% ± 0.4% (n=3)
  NL→IT: -25.0%, 28/36 (n=6v6), p=0.066 *
  NL→CR: -65.7%, 18/18 (n=6v3), p=0.012 **
  IT→IA: -32.9%, 23/30 (n=6v5), p=0.089 *
  IA→AR: +11.2%, 10/15 (n=5v3), p=0.286

--- CD8_T ---
  NL: 26.3% ± 11.7% (n=6)
  IT: 22.5% ± 10.8% (n=6)
  IA: 34.7% ± 9.1% (n=5)
  AR: 34.5% ± 1.3% (n=3)
  CR: 23.1% ± 7.3% (n=3)
  NL→IT: -14.2%, 25/36 (n=6v6), p=0.155
  NL→CR: -12.1%, 11/18 (n=6v3), p=0.357
  IT→IA: +54.0%, 26/30 (n=6v5), p=0.026 **
  IA→AR: -0.7%, 9/15 (n=5v3), p=0.393

--- CD4_T ---
  NL: 17.8% ± 6.6% (n=6)
  IT: 26.7% ± 12.4% (n=6)
  IA: 27.6% ± 5.3% (n=5)
  AR: 30.1% ± 3.1% (n=3)
  CR: 33.2% ± 3.2% (n=3)
  NL→IT: +49.6%, 24/36 (n=6v6), p=0.197
  NL→CR: +86.0%, 18/18 (n=6v3), p=0.012 **
  IT→IA: +3.6%, 16/30 (n=6v5), p=0.465
  IA→AR: +8.8%, 9/15 (n=5v3), p=0.393

--- Myeloid ---
  NL: 12.0% ± 20.4% (n=6)
  IT: 13.9% ± 8.0% (

In [6]:
# ============================================================
# 4. PlasmaB/B RATIO
# ============================================================
print("\n" + "="*60)
print("PlasmaB/B RATIO — FIXED")
print("="*60)

b_counts = adata.obs[adata.obs['major_lineage']=='B'].groupby('sample_id').size().to_dict()
pb_counts = adata.obs[adata.obs['major_lineage']=='PlasmaB'].groupby('sample_id').size().to_dict()

ratio_records = []
for sid in totals:
    b = b_counts.get(sid, 0)
    pb = pb_counts.get(sid, 0)
    if b + pb > 0:
        ratio_records.append({
            'sample_id': sid, 'Stage': stage_map[sid],
            'b_cells': b, 'pb_cells': pb,
            'ratio': pb / (b + pb)
        })
ratio_df = pd.DataFrame(ratio_records)

for stage in STAGE_ORDER:
    sd = ratio_df[ratio_df['Stage']==stage]['ratio']
    if len(sd) > 0:
        print(f"  {stage}: {sd.mean():.4f} ± {sd.std():.4f} (n={len(sd)})")

for sa, sb in [('NL','IT'),('NL','CR'),('IT','CR')]:
    va = ratio_df[ratio_df['Stage']==sa]['ratio'].values
    vb = ratio_df[ratio_df['Stage']==sb]['ratio'].values
    r = donor_test(va, vb)
    sig = " **" if r['p']<0.05 else " *" if r['p']<0.1 else ""
    print(f"  {sa}→{sb}: {r['consist']}/{r['total']} (n={r['n_a']}v{r['n_b']}), p={r['p']:.3f}{sig}")



PlasmaB/B RATIO — FIXED
  NL: 0.3035 ± 0.2902 (n=6)
  IT: 0.0922 ± 0.0359 (n=6)
  IA: 0.0852 ± 0.0586 (n=5)
  AR: 0.1085 ± 0.0634 (n=3)
  CR: 0.0476 ± 0.0034 (n=3)
  NL→IT: 23/36 (n=6v6), p=0.242
  NL→CR: 12/18 (n=6v3), p=0.274
  IT→CR: 18/18 (n=6v3), p=0.012 **


In [7]:
# ============================================================
# 5. SAVE
# ============================================================
pd.DataFrame(all_stats).to_csv(os.path.join(CSV_DIR,'lineage_proportions_statistics_FIXED.csv'), index=False)
ratio_df.to_csv(os.path.join(CSV_DIR,'bcell_plasmaB_ratio_donor_FIXED.csv'), index=False)
print(f"\n✅ Saved to: {CSV_DIR}")
print("DONE.")




✅ Saved to: /content/drive/MyDrive/ITLAS/results/task3_donor_validation/csv
DONE.
